# SSB StatBank → Standardize Layer

Leser JSON-filer fra Landing layer og konverterer til flate Delta-tabeller i Standardize layer.

## Hva notebooken gjør
- Leser `statbank_staging.pipeline.ssb_load_queue` for å vite hvilke tabeller som skal prosesseres
- For hver tabell: finn nyeste snapshot i Landing, les alle periode-filer
- Parser JSON-Stat2 til flat Spark DataFrame med `_code`/`_label`-kolonner
- Beriker med kommunekorrespondanse (gjeldende kode + `er_akershus`-flagg)
- Kjører datakvalitetssjekker
- Skriver til Delta-tabell partisjonert på `year`
- Kjører `ssb_refine_{table_id}` hvis den finnes (tabellspesifikk raffinering)

## Kjøring
Kjøres fra Fabric Pipeline etter `04_ssb_ingest_landing`.
Parameteren `QUEUE_TABLE` settes av pipeline.

---------


In [1]:
# =====================================================================
# PARAMETERE (overstyres av Fabric Pipeline)
# Avgjør hvilke tabeller som skal standardiseres i denne kjøringen, og om
# skriving til Silver-tabellene skal legge til (append) eller erstatte
# (overwrite) alt innhold.

QUEUE_TABLE          = "statbank_staging.pipeline.ssb_load_queue"   # eller statbank_staging.pipeline.ssb_load_queue_critical
SNAPSHOT_DATE        = None               # None = bruk nyeste snapshot
WRITE_MODE           = "append"        # overwrite eller append

# Midlertidig – begrens til én tabell fra ssb_config for testing
#QUEUE_TABLE          = "statbank_staging.pipeline.ssb_config"
DEBUG_TABLE_ID = None   # Sett til None for å kjøre alle


StatementMeta(, eb88c85d-40bd-4a35-85ce-1ceb6f269737, 3, Finished, Available, Finished, False)

In [2]:
# =====================================================================
# IMPORTS
# Standard oppsett – kobler til Spark og gjør PySpark-verktøyene som brukes
# gjennom hele notebooken tilgjengelig.
from __future__ import annotations

import itertools
import json
import re
import time
from datetime import date, datetime, timezone
from typing import Dict, List, Optional, Tuple

from pyspark.sql import DataFrame, Row, SparkSession
from pyspark.sql import functions as F
from pyspark.sql.types import StructType, StructField, StringType, DoubleType, IntegerType, BooleanType, TimestampType, DateType

spark = SparkSession.builder.appName("SSBStandardize").getOrCreate()

from notebookutils import mssparkutils

print("Imports OK")


StatementMeta(, eb88c85d-40bd-4a35-85ce-1ceb6f269737, 4, Finished, Available, Finished, False)

Imports OK | Fabric: True


In [3]:
# =====================================================================
# KONFIGURASJON
# Faste innstillinger: hvor i lakehouset de rå JSON-filene fra Landing
# ligger, hvilket schema de ferdige Silver-tabellene skal skrives til, og
# noen terskelverdier (hvor høy null-andel som regnes som et datakvalitets-
# problem, og hvor mange celler en tabell må ha før utflatingen distribueres
# over klyngen i stedet for å kjøres lokalt).
LAKEHOUSE_ROOT     = "Files"
ZONE_LANDING       = "landing"
DATA_SOURCE        = "ssb"
DATA_PRODUCT       = "statbank"
STANDARDIZE_SCHEMA = "statbank_staging.ssb"   # Måltabell-schema i Standardize layer (lakehouse-kvalifisert)
MAX_NULL_RATE      = 0.80    # SSB krysstabell-data har naturlig høy null-rate
DISTRIBUTE_CELL_THRESHOLD = 500_000  # celler – over denne grensen distribueres
                                     # JSON-Stat2-utflating over klyngen (se parse_jsonstat2_to_dataframe)

# Dimensjonstabell for kommunekorrespondanser
DIM_KOMMUNE_PATH = "abfss://AFKA-DataEng@onelake.dfs.fabric.microsoft.com/statbank_staging.lakehouse/Tables/kodeverk/dim_regionkoder_historikk"

print("Konfigurasjon OK")

StatementMeta(, eb88c85d-40bd-4a35-85ce-1ceb6f269737, 5, Finished, Available, Finished, False)

Konfigurasjon OK


In [4]:
# =====================================================================
# FABRIC FILESYSTEM HELPERS
# Verktøy for å finne frem i Landing layer: FabricFS gir enkel lese-tilgang
# til filene, mens get_latest_snapshot_path og get_period_files finner
# henholdsvis nyeste innlastingsmappe og hvilke periode-filer (år/kvartal/
# måned osv.) som finnes der for en gitt tabell.

class FabricFS:
    """Filoperasjoner via mssparkutils."""

    @staticmethod
    def normalize_path(path: str) -> str:
        if path.startswith("Files/"):
            return path
        if path.startswith("/lakehouse/default/Files/"):
            return path.replace("/lakehouse/default/", "")
        return path

    @staticmethod
    def exists(path: str) -> bool:
        path = FabricFS.normalize_path(path)
        try:
            mssparkutils.fs.ls(path)
            return True
        except Exception:
            return False

    @staticmethod
    def read_json(path: str) -> dict:
        """
        Les JSON – bruker Spark for store filer, fallback til mssparkutils.
        """
        path = FabricFS.normalize_path(path)
        try:
            # Use Spark if file is large enough for distributed reading
            rdd = spark.sparkContext.textFile(path)
            return json.loads("\n".join(rdd.collect()))
        except Exception:
            pass
        # For små filer eller hvis Spark feiler, fallback til mssparkutils
        content = mssparkutils.fs.head(path, 500_000_000)
        return json.loads(content)

    @staticmethod
    def list_subdirs(path: str) -> list:
        """List undermapper, returner mappenavn (ikke full sti)."""
        path = FabricFS.normalize_path(path)
        try:
            return [item.name for item in mssparkutils.fs.ls(path) if item.isDir]
        except Exception:
            return []


def get_latest_snapshot_path(table_id: str) -> Optional[str]:
    """
    Finn nyeste snapshot-mappe for en tabell.
    Returnerer full relativ sti, eller None hvis ingen snapshot finnes.
    """
    landing_base = f"{LAKEHOUSE_ROOT}/{ZONE_LANDING}/{DATA_SOURCE}/{DATA_PRODUCT}/{table_id}"
    subdirs = FabricFS.list_subdirs(landing_base)
    snapshots = sorted(
        [d for d in subdirs if d.startswith("snapshot_date=")],
        reverse=True,
    )
    if not snapshots:
        return None
    target = SNAPSHOT_DATE or snapshots[0].replace("snapshot_date=", "")
    for snap in snapshots:
        if snap == f"snapshot_date={target}":
            return f"{landing_base}/{snap}"
    return None

def get_period_files(snapshot_path: str, table_id: str) -> list:
    """
    Finn alle periode-filer i et snapshot.
    Returnerer liste av (period, file_path).
    Avhenger ikke av manifest – leser mappestrukturen direkte.
    """
    period_dirs = FabricFS.list_subdirs(snapshot_path)
    results = []
    for d in sorted(period_dirs):
        if not d.startswith("period="):
            continue
        period = d.replace("period=", "")
        file_path = f"{snapshot_path}/{d}/ssb_{table_id}_{period}.json"
        if FabricFS.exists(file_path):
            results.append((period, file_path))
    return results

print("FabricFS OK")


StatementMeta(, eb88c85d-40bd-4a35-85ce-1ceb6f269737, 6, Finished, Available, Finished, False)

FabricFS OK


In [5]:
# =====================================================================
# TIDSKODE PARSING
# SSB bruker mange ulike skrivemåter for tidsperioder (2024K1 for kvartal,
# 2024M03 for måned, 2024U12 for uke, osv.). Denne funksjonen tolker koden
# og gir tilbake år/kvartal/måned/uke som egne kolonner, slik at dataene
# blir enklere å filtrere og analysere i Silver-tabellen.

def parse_period_to_parts(period: str) -> Dict:
    """
    Parser tidskode til year, quarter, month, week, period_type.
    Brukes for partisjonering og analyse.
    """
    period = str(period).strip()

    m = re.match(r"(\d{4})[KkQq](\d{1})", period)
    if m:
        return {"year": int(m.group(1)), "quarter": int(m.group(2)),
                "month": None, "week": None, "period_type": "quarter"}

    m = re.match(r"(\d{4})[Mm](\d{2})", period)
    if m:
        return {"year": int(m.group(1)), "quarter": None,
                "month": int(m.group(2)), "week": None, "period_type": "month"}

    m = re.match(r"(\d{4})[UuWw](\d{2})", period)
    if m:
        return {"year": int(m.group(1)), "quarter": None,
                "month": None, "week": int(m.group(2)), "period_type": "week"}

    m = re.match(r"(\d{4})[Hh](\d{1})", period)
    if m:
        return {"year": int(m.group(1)), "quarter": None, "month": None,
                "week": None, "period_type": "half_year"}

    if "-" in period:
        year_str = period.split("-")[0]
    else:
        year_str = period

    m = re.search(r"(\d{4})", year_str)
    return {"year": int(m.group(1)) if m else None,
            "quarter": None, "month": None, "week": None, "period_type": "year"}

print("Tidskode-parsing OK")


StatementMeta(, eb88c85d-40bd-4a35-85ce-1ceb6f269737, 7, Finished, Available, Finished, False)

Tidskode-parsing OK


In [6]:
# =====================================================================
# GEO-NIVÅ KLASSIFISERING
# Ut fra lengden på en regionkode gjettes det geografiske nivået –
# nasjonalt, fylke, kommune eller bydel. Brukes til å merke rader slik at
# man senere kan filtrere data på riktig geografisk detaljnivå.

def classify_geo_level(region_code: str) -> Optional[str]:
    """
    Klassifiser geografisk niva basert pa region_code-lengde.
    1 siffer  -> N (Nasjonalt)
    2 siffer  -> F (Fylke)
    4 siffer  -> K (Kommune)
    6 siffer  -> B (Bydel)
    """
    if not region_code:
        return None
    code = str(region_code).strip()
    if not code.isdigit():
        if code and code[-1].isalpha():
            code = code[:-1]
        else:
            return None
    return {1: "N", 2: "F", 4: "K", 6: "B"}.get(len(code))

print("Geo-niva OK")

StatementMeta(, eb88c85d-40bd-4a35-85ce-1ceb6f269737, 8, Finished, Available, Finished, False)

Geo-niva OK


In [7]:
# =====================================================================
# JSON-STAT2 PARSER
# Dette er hjertet i notebooken: SSB leverer data i et format (JSON-Stat2)
# der alle tall ligger i én lang liste, og man må selv regne ut hvilken
# kombinasjon av dimensjoner (region, kjønn, alder, år, osv.) hvert enkelt
# tall tilhører. Funksjonene under gjør nettopp det, og bygger om resultatet
# til en ryddig tabell med én kolonne per dimensjon – det formatet resten av
# pipelinen (og til slutt Power BI/rapportering) kan bruke direkte.


def _infer_spark_type(value):
    """Enkel type-inferering for Python-verdier → Spark-type.

    Brukes for å unngå [CANNOT_DETERMINE_TYPE] når alle verdier i en kolonne
    er NULL (Spark klarer da ikke å inferere type automatisk).
    """
    if value is None:
        return StringType()
    if isinstance(value, bool):
        return BooleanType()
    if isinstance(value, int):
        return IntegerType()
    if isinstance(value, float):
        return DoubleType()
    if isinstance(value, (datetime,)):
        return TimestampType()
    return StringType()


def _build_schema_from_records(records: list) -> StructType:
    """Bygg StructType fra records.

    - Kjente felt (table_id, value osv.) får hardkodede typer
    - Dynamiske dimensjonskolonner ({dim}_code / {dim}_label, geo_level)
      er alltid StringType
    - _infer_spark_type brukes kun som siste fallback for ukjente felt
    """
    if not records:
        return StructType([])

    KNOWN_FIELDS = {
        "table_id":       StringType(),
        "snapshot_date":  StringType(),
        "period":         StringType(),
        "value":          DoubleType(),
        "geo_level":      StringType(),
        "ssb_updated_at": DateType(),
        "ssb_title":      StringType(),
    }

    all_keys = set()
    for r in records:
        all_keys.update(r.keys())

    fields = []
    for key in sorted(all_keys):
        if key in KNOWN_FIELDS:
            spark_type = KNOWN_FIELDS[key]
        elif key.endswith("_code") or key.endswith("_label"):
            spark_type = StringType()
        else:
            sample = next((r[key] for r in records if r.get(key) is not None), None)
            spark_type = _infer_spark_type(sample)
        fields.append(StructField(key, spark_type, nullable=True))

    return StructType(fields)


def _build_record(
    flat_idx: int,
    values: list,
    sizes: list,
    dim_categories: list,
    table_id: str,
    snapshot_date: str,
    period: str,
    ssb_updated_at,
    ssb_title,
) -> Optional[Dict]:
    """
    Bygg én flat rad for celle nr. flat_idx i JSON-Stat2 sin value-array.

    Dekomponerer flat_idx til per-dimensjon-indekser med nøyaktig samme
    rekkefølge som itertools.product(*[range(s) for s in sizes]) gir
    (siste dimensjon varierer raskest – standard rad-major/mixed-radix-
    dekomponering: idx[k] = (flat_idx // prod(sizes[k+1:])) % sizes[k]).
    Brukes uendret av både den enkle og den distribuerte utflatingsveien
    under, slik at de to garantert gir identisk resultat.
    """
    if flat_idx >= len(values):
        return None

    raw_val = values[flat_idx]
    record = {
        "table_id":       table_id,
        "snapshot_date":  snapshot_date,
        "ssb_updated_at": ssb_updated_at,
        "ssb_title":      ssb_title,
        "period":         period,
        "value":          float(raw_val) if raw_val is not None else None,
    }

    idx_per_dim = [0] * len(sizes)
    remainder = flat_idx
    for k in range(len(sizes) - 1, -1, -1):
        idx_per_dim[k] = remainder % sizes[k]
        remainder //= sizes[k]

    region_code = None
    for dim_idx, (col, cats, labels, is_geo) in enumerate(dim_categories):
        code  = cats[idx_per_dim[dim_idx]]
        label = labels.get(code, code)
        record[f"{col}_code"]  = code
        record[f"{col}_label"] = label
        if is_geo:
            region_code = code

    record["geo_level"] = classify_geo_level(region_code) if region_code else None
    return record


def parse_jsonstat2_to_dataframe(
    json_data: dict,
    period: str,
    snapshot_date: str,
    table_id: str,
) -> DataFrame:
    """Parser JSON-Stat2 til flat Spark DataFrame.

    Hver dimensjon far to kolonner: {dim}_code og {dim}_label.
    Tidsdimensjonen far ekstra kolonner: year, quarter, month, week, period_type.
    Geografisk dimensjon far geo_level (N/F/K/B).

    For tabeller med mange celler (mange dimensjoner × brede kodelister,
    ev. kombinert med lang lookback) bygges radene distribuert over klyngen
    (sc.parallelize + broadcast) i stedet for som én stor Python-liste på
    driveren – unngår driver-OOM for brede krysstabeller. Terskelen styres
    av DISTRIBUTE_CELL_THRESHOLD. For de aller fleste (mindre) tabeller
    brukes fortsatt den raske driver-lokale veien uendret.
    """
    from datetime import date as _date
    dimensions = json_data.get("dimension", {})
    dim_ids    = list(dimensions.keys())
    values     = json_data.get("value", [])
    ssb_updated_at = (
        _date.fromisoformat(json_data["updated"][:10])
        if json_data.get("updated") else None
    )
    ssb_title = json_data.get("label")

    sizes = [len(dimensions[d]["category"]["index"]) for d in dim_ids]
    total_cells = 1
    for s in sizes:
        total_cells *= s

    dim_categories = []
    for dim_id in dim_ids:
        dim_data = dimensions[dim_id]
        cats   = list(dim_data["category"]["index"].keys())
        labels = dim_data["category"]["label"]
        col    = dim_id.lower().replace(" ", "_")
        is_geo = "region" in col or "kommune" in col
        dim_categories.append((col, cats, labels, is_geo))

    if total_cells <= DISTRIBUTE_CELL_THRESHOLD:
        # Rask driver-lokal vei – dekker de aller fleste SSB-tabeller
        records = []
        for flat_idx in range(min(total_cells, len(values))):
            record = _build_record(
                flat_idx, values, sizes, dim_categories,
                table_id, snapshot_date, period, ssb_updated_at, ssb_title,
            )
            if record is not None:
                records.append(record)
        schema = _build_schema_from_records(records)
        df = spark.createDataFrame(records, schema=schema)
    else:
        # Distribuert vei – store/brede tabeller
        print(f"   {total_cells:,} celler (> {DISTRIBUTE_CELL_THRESHOLD:,}) – distribuerer utflating over klyngen")

        sample = []
        for flat_idx in range(min(200, total_cells, len(values))):
            record = _build_record(
                flat_idx, values, sizes, dim_categories,
                table_id, snapshot_date, period, ssb_updated_at, ssb_title,
            )
            if record is not None:
                sample.append(record)
        schema = _build_schema_from_records(sample)

        values_bc         = spark.sparkContext.broadcast(values)
        dim_categories_bc = spark.sparkContext.broadcast(dim_categories)
        num_partitions = max(1, min(400, (total_cells // 50_000) + 1))

        def _build_row(flat_idx: int):
            rec = _build_record(
                flat_idx, values_bc.value, sizes, dim_categories_bc.value,
                table_id, snapshot_date, period, ssb_updated_at, ssb_title,
            )
            return Row(**rec) if rec is not None else None

        flat_rdd = (
            spark.sparkContext.parallelize(range(total_cells), num_partitions)
            .map(_build_row)
            .filter(lambda r: r is not None)
        )
        df = spark.createDataFrame(flat_rdd, schema=schema)

    tp = parse_period_to_parts(period)
    df = (
        df
        .withColumn("year",        F.lit(tp["year"]).cast(IntegerType()))
        .withColumn("quarter",     F.lit(tp["quarter"]).cast(IntegerType()))
        .withColumn("month",       F.lit(tp["month"]).cast(IntegerType()))
        .withColumn("week",        F.lit(tp["week"]).cast(IntegerType()))
        .withColumn("period_type", F.lit(tp["period_type"]))
        .withColumn("processed_at", F.current_timestamp())
    )
    return df

print("JSON-Stat2 parser OK")


StatementMeta(, eb88c85d-40bd-4a35-85ce-1ceb6f269737, 9, Finished, Available, Finished, False)

JSON-Stat2 parser OK


In [8]:
# =====================================================================
# KOMMUNE-BERIKELSE
# Kommune- og regiongrenser i Norge endres innimellom (sammenslåinger,
# navnebytter). Denne funksjonen oversetter de historiske kommunekodene som
# står i SSB-dataene til dagens gjeldende kommunenummer, ved å slå opp i en
# egen oversiktstabell (dim_regionkoder_historikk). Merker samtidig av
# hvilke rader som gjelder Akershus.

def enrich_with_kommune_2024(df: DataFrame) -> DataFrame:
    """
    Beriker med oppdatert kommunenummer fra dimensjonstabellen.
    Left join på region_code -> historisk_kode.

    Regionskolonnen kan hete region_code, kokkommuneregion0000_code o.l.
    – funksjonen finner den første _code-kolonnen som inneholder
    'region' eller 'kommune' i navnet.

    Legger til:
      - region_code_gjeldende  (gjeldende kommunenummer)
      - fylke_kode             (fra dimensjonstabellen)
      - er_akershus            (True hvis fylke_kode starter med "32")
    Tabeller uten region-dimensjon returneres med alle tre som None/False.
    """
    region_col = next(
        (c for c in df.columns
         if c.endswith("_code") and any(kw in c for kw in ("region", "kommune"))),
        None,
    )

    if region_col is None:
        print("   Ingen region/kommune-kolonne – hopper over kommune-berikelse")
        return (
            df
            .withColumn("region_code_gjeldende", F.lit(None).cast(StringType()))
            .withColumn("fylke_kode",             F.lit(None).cast(StringType()))
            .withColumn("er_akershus",            F.lit(False))
        )

    print(f"   Bruker '{region_col}' som regionskolonne")
    try:
        df_dim = (
            spark.read.format("delta").load(DIM_KOMMUNE_PATH)
            .select("historisk_kode", "gjeldende_kode", "fylke_kode")
            .distinct()
        )
        df_enriched = (
            df
            .join(df_dim, df[region_col] == df_dim["historisk_kode"], "left")
            .drop("historisk_kode")
            .withColumnRenamed("gjeldende_kode", "region_code_gjeldende")
            .withColumn(
                "er_akershus",
                F.col("region_code_gjeldende").startswith("32")
            )
        )
        stats = df_enriched.agg(
            F.count("*").alias("total"),
            F.sum(F.when(F.col("region_code_gjeldende").isNotNull(), 1).otherwise(0)).alias("mapped"),
        ).collect()[0]
        total  = stats["total"]
        mapped = stats["mapped"]
        print(f"   Kommune-berikelse: {mapped:,}/{total:,} rader mappet ({mapped/total:.1%})")
        return df_enriched

    except Exception as e:
        print(f"   ADVARSEL: Kommune-berikelse feilet ({e}) – fortsetter uten")
        return (
            df
            .withColumn("region_code_gjeldende", F.lit(None).cast(StringType()))
            .withColumn("fylke_kode",             F.lit(None).cast(StringType()))
            .withColumn("er_akershus",            F.lit(False))
        )

print("Kommune-berikelse OK")

StatementMeta(, eb88c85d-40bd-4a35-85ce-1ceb6f269737, 10, Finished, Available, Finished, False)

Kommune-berikelse OK


In [9]:
# =====================================================================
# DATAKVALITETSSJEKKER
# En rask helsesjekk av dataene før de skrives til Silver: hvor mange rader
# er det, hvor stor andel mangler verdi, finnes det duplikater, og hvilket
# årsspenn dekker dataene. Stopper ikke kjøringen, men varsler i loggen hvis
# noe ser mistenkelig ut.

def run_data_quality_checks(df: DataFrame) -> Dict:
    """Grunnleggende datakvalitetssjekker.

    Samler total, nulls og årsintervall i én enkelt Spark-action
    for å unngå unødvendige fulle tabellscanninger.
    """
    stats = df.agg(
        F.count("*").alias("total"),
        F.sum(F.when(F.col("value").isNull(), 1).otherwise(0)).alias("nulls"),
        F.min("year").alias("year_min"),
        F.max("year").alias("year_max"),
    ).collect()[0]

    total     = stats["total"]
    nulls     = stats["nulls"]
    null_rate = nulls / total if total > 0 else 0
    year_min  = stats["year_min"]
    year_max  = stats["year_max"]

    key_cols   = [c for c in df.columns if c.endswith("_code")] + ["period"]
    duplicates = (
        df.groupBy(key_cols).count().filter(F.col("count") > 1).count()
        if key_cols else 0
    )

    checks = {
        "total_rows":    total,
        "null_values":   nulls,
        "null_rate":     round(null_rate, 4),
        "duplicates":    duplicates,
        "year_min":      year_min,
        "year_max":      year_max,
        "has_issues":    null_rate > MAX_NULL_RATE or duplicates > 0,
    }

    print(f"   Rader: {total:,}  |  Null: {nulls:,} ({null_rate:.1%})  |  "
          f"Duplikater: {duplicates}  |  Ar: {year_min}-{year_max}")
    if checks["has_issues"]:
        print("   ADVARSEL: Datakvalitetsproblemer funnet")

    return checks

print("Datakvalitet OK")

StatementMeta(, eb88c85d-40bd-4a35-85ce-1ceb6f269737, 11, Finished, Available, Finished, False)

Datakvalitet OK


In [10]:
# =====================================================================
# SKRIV TIL SILVER
# Den siste delen av reisen – skriver den ferdig standardiserte og berikede
# tabellen til Delta-tabellen i Standardize (Silver) layer. Velger
# automatisk mellom å skrive alt på nytt (full overwrite) eller kun
# oppdatere de siste periodene (MERGE), avhengig av om tabellen finnes fra
# før og om den er lastet tidligere – se docstringen under for detaljene.

def write_to_delta(df: DataFrame, target_table: str, last_loaded_timestamp) -> None:
    """
    Skriv til Delta-tabell i Standardize layer, partisjonert på year.

    Skrivestrategi (prioritert rekkefølge):
      1. WRITE_MODE == "overwrite"          → full overwrite (alle perioder)
      2. Tabell finnes ikke                 → full overwrite (første gang)
      3. last_loaded_timestamp er None/NULL → full overwrite (aldri lastet)
      4. Ellers                             → atomisk MERGE på de to siste periodene
    """
    from delta.tables import DeltaTable

    meta_cols = ["table_id", "period", "period_type",
                 "year", "quarter", "month", "week", "ssb_title", "ssb_updated_at", "snapshot_date"]
    dim_cols  = sorted([c for c in df.columns if c.endswith(("_code", "_label"))])
    val_cols  = [c for c in [
        "value", "geo_level", "fylke_kode", "er_akershus",
        "region_code_gjeldende", "processed_at"
    ] if c in df.columns]
    ordered = [c for c in meta_cols + dim_cols + val_cols if c in df.columns]
    df      = df.select(ordered)

    table_exists = spark.catalog.tableExists(target_table)

    full_load = (
        WRITE_MODE == "overwrite"
        or not table_exists
        or last_loaded_timestamp is None
    )

    if full_load:
        if WRITE_MODE == "overwrite":
            aarsak = "WRITE_MODE=overwrite"
        elif not table_exists:
            aarsak = "tabell finnes ikke"
        else:
            aarsak = "last_loaded_timestamp er NULL"
        print(f"   Skrivemodus: full overwrite ({aarsak})")
        (
            df.write
            .format("delta")
            .mode("overwrite")
            .partitionBy("year")
            .option("overwriteSchema", "true")
            .option("mergeSchema", "true")
            .saveAsTable(target_table)
        )
    else:
        # Atomisk MERGE på de to siste periodene – unngår datatap ved feil
        alle_perioder  = sorted([r["period"] for r in df.select("period").distinct().collect()])
        siste_perioder = alle_perioder[-2:]
        df_skriv       = df.filter(F.col("period").isin(siste_perioder))

        print(f"   Skrivemodus: MERGE siste 2 perioder ({', '.join(siste_perioder)})")

        code_cols  = [c for c in df_skriv.columns if c.endswith("_code")]
        merge_cond = " AND ".join(
            ["target.table_id = source.table_id", "target.period = source.period"]
            + [f"target.{c} = source.{c}" for c in code_cols]
        )

        (
            DeltaTable.forName(spark, target_table)
            .alias("target")
            .merge(df_skriv.alias("source"), merge_cond)
            .whenMatchedUpdateAll()
            .whenNotMatchedInsertAll()
            .execute()
        )

    try:
        spark.sql(f"OPTIMIZE {target_table} ZORDER BY (period, region_code_gjeldende)")
        print("   OPTIMIZE (ZORDER) fullført")
    except Exception as e:
        print(f"   ADVARSEL: OPTIMIZE feilet (ikke kritisk): {e}")

    print(f"   Skrevet til {target_table}")


def set_table_metadata(
    target_table: str,
    table_id: str,
    table_name: str,
    category: str,
    frequency: str,
) -> None:
    """
    Setter tabellbeskrivelse og litt ekstra metadata (kategori, frekvens,
    tabellnummer, lenke til SSB, tidspunkt for siste standardisering) som
    egenskaper på Silver-tabellen. Gjør at noen som blar i lakehouset – eller
    kobler Power BI til SQL-endepunktet – kan se hva tabellen faktisk
    inneholder uten å måtte slå opp i ssb_config/ssb_metadata først.

    Ikke kritisk for pipelinen – feiler den, logges bare en advarsel.
    """
    def _esc(v) -> str:
        return (v or "").replace("'", "''")

    comment = _esc(table_name) or f"SSB-tabell {table_id}"
    now = datetime.now(timezone.utc).isoformat()

    try:
        spark.sql(f"COMMENT ON TABLE {target_table} IS '{comment}'")
        spark.sql(f"""
            ALTER TABLE {target_table} SET TBLPROPERTIES (
                'ssb.tabellnummer'         = '{table_id}',
                'ssb.kategori'             = '{_esc(category)}',
                'ssb.oppdateringsfrekvens' = '{_esc(frequency)}',
                'ssb.sist_standardisert'   = '{now}',
                'ssb.kilde_url'            = 'https://www.ssb.no/statbank/table/{table_id}'
            )
        """)
        print("   Tabellmetadata satt (beskrivelse + kategori/frekvens/kilde)")
    except Exception as e:
        print(f"   ADVARSEL: Kunne ikke sette tabellmetadata (ikke kritisk): {e}")


print("Skriving OK")

StatementMeta(, eb88c85d-40bd-4a35-85ce-1ceb6f269737, 12, Finished, Available, Finished, False)

Skriving OK


In [11]:
# =====================================================================
# OPPDATER LAST_LOADED_TIMESTAMP I SSB_CONFIG
# Etter at én eller flere tabeller er ferdig skrevet til Silver, skrives
# dagens tidspunkt til last_loaded_timestamp i ssb_config. Dette er ulikt
# last_downloaded_timestamp (satt av 04) – denne settes først når tabellen
# faktisk er ferdig standardisert helt til slutt i kjeden.

def update_last_loaded_timestamp(table_ids: List[str], timestamp: str) -> None:
    """
    Oppdater last_loaded_timestamp i ssb_config for alle tabeller
    som ble vellykket prosessert i denne kjøringen.
    Kjøres etter at alle tabeller er ferdig – ikke per tabell.
    """
    if not table_ids:
        print("   Ingen tabeller å oppdatere i ssb_config")
        return
    ids_str = ", ".join(f"\'{tid}\'" for tid in table_ids)
    spark.sql(f"""
        UPDATE statbank_staging.pipeline.ssb_config
        SET last_loaded_timestamp = \'{timestamp}\'
        WHERE table_id IN ({ids_str})
    """)
    print(f"✅ last_loaded_timestamp oppdatert for {len(table_ids)} tabeller: {', '.join(table_ids)}")

print("update_last_loaded_timestamp OK")


StatementMeta(, eb88c85d-40bd-4a35-85ce-1ceb6f269737, 13, Finished, Available, Finished, False)

update_last_loaded_timestamp OK


In [12]:
# =====================================================================
# PROSESSER ÉN TABELL
# Samler alt som skjer med én SSB-tabell fra start til slutt: finn dataene
# i Landing, les og flat ut hver periode, berik med kommunekoder, kjør
# kvalitetssjekk, og skriv resultatet til Silver. Underveis sjekkes det også
# om en tabellspesifikk "raffinerings"-notebook (ssb_refine_<tabellnr>)
# finnes og bør kjøres etterpå for ekstra tilpasning av akkurat den tabellen.

def process_table(table_id: str) -> dict:
    """
    Les alle perioder fra nyeste snapshot og skriv til Standardize.

    Skrivestrategi avgjøres av write_to_delta basert på:
      - WRITE_MODE-parameter
      - om tabellen eksisterer i katalogen
      - last_loaded_timestamp fra ssb_config (None → full load)
    """
    target_table = f"{STANDARDIZE_SCHEMA}.statbank_{table_id}"
    print(f"\n{'='*60}")
    print(f"Tabell: {table_id}  ->  {target_table}")
    print(f"{'='*60}")

    result = {
        "status":       "unknown",
        "table_id":     table_id,
        "target_table": target_table,
        "rows":         0,
        "periods":      0,
        "error":        None,
    }
    try:
        # Finn snapshot
        snapshot_path = get_latest_snapshot_path(table_id)
        if not snapshot_path:
            raise ValueError(f"Ingen snapshot funnet for {table_id} i Landing layer")
        snapshot_date = snapshot_path.split("snapshot_date=")[-1]
        print(f"Snapshot: {snapshot_date}")

        # Hent konfigurasjon fra ssb_config (lookback_periods, last_loaded_timestamp,
        # samt table_name/category/frequency – brukes til å merke Silver-tabellen
        # med lesbar metadata i set_table_metadata())
        config_row = (
            spark.table("statbank_staging.pipeline.ssb_config")
            .filter(F.col("table_id") == table_id)
            .select("table_name", "category", "frequency", "lookback_periods", "last_loaded_timestamp")
            .collect()
        )
        lookback              = config_row[0]["lookback_periods"]       if config_row else None
        last_loaded_timestamp = config_row[0]["last_loaded_timestamp"]  if config_row else None
        config_table_name     = config_row[0]["table_name"]             if config_row else None
        config_category       = config_row[0]["category"]               if config_row else None
        config_frequency      = config_row[0]["frequency"]              if config_row else None

        # Sporing: sammenlign hvilken 03-kjøring (check_timestamp) som faktisk ble
        # hentet til Landing av 04, mot hva som står i køen nå – avdekker race
        # conditions der 03 har kjørt på nytt mens 04/05 fortsatt jobbet.
        try:
            ingest_manifest = FabricFS.read_json(f"{snapshot_path}/manifest.json")
            ingest_check_timestamp = ingest_manifest.get("queue_check_timestamp")
        except Exception:
            ingest_check_timestamp = None

        if ingest_check_timestamp:
            try:
                queue_row = (
                    spark.table(QUEUE_TABLE)
                    .filter(F.col("table_id") == table_id)
                    .select("check_timestamp")
                    .collect()
                )
                current_check_timestamp = queue_row[0]["check_timestamp"] if queue_row else None
                if current_check_timestamp and current_check_timestamp != ingest_check_timestamp:
                    print(
                        f"⚠️  ADVARSEL: Køen er endret siden 04 hentet denne tabellen "
                        f"(04 brukte: {ingest_check_timestamp}, køen nå: {current_check_timestamp}) "
                        f"– mulig race condition mot 03"
                    )
            except Exception:
                pass  # QUEUE_TABLE mangler kanskje check_timestamp-kolonnen (f.eks. debug mot ssb_config)

        # Finn periode-filer direkte fra mappestruktur (uavhengig av manifest)
        period_files = get_period_files(snapshot_path, table_id)

        # Begrens perioder ved normal drift; last alt ved full load
        table_exists = spark.catalog.tableExists(target_table)
        full_load = (
            WRITE_MODE == "overwrite"
            or not table_exists
            or last_loaded_timestamp is None
        )

        if lookback and not full_load:
            period_files = period_files[-lookback:]
            print(f"Lookback: siste {lookback} perioder ({period_files[0][0]} -> {period_files[-1][0]})")
        else:
            print(f"Lookback: deaktivert (full_load={full_load}) – laster alle {len(period_files)} perioder")

        if not period_files:
            raise ValueError(f"Ingen periode-filer funnet i {snapshot_path}")
        print(f"Perioder: {len(period_files)} ({period_files[0][0]} -> {period_files[-1][0]})")

        # Les og parser alle perioder
        dfs = []
        for idx, (period, file_path) in enumerate(period_files, 1):
            try:
                json_data = FabricFS.read_json(file_path)
                df_period = parse_jsonstat2_to_dataframe(
                    json_data, period, snapshot_date, table_id
                )
                dfs.append(df_period)
                if idx % 10 == 0 or idx == len(period_files):
                    print(f"   [{idx}/{len(period_files)}] {period} OK")
            except Exception as e:
                print(f"   [{idx}/{len(period_files)}] {period} FEIL: {e}")
                continue

        if not dfs:
            raise ValueError("Ingen perioder kunne prosesseres")

        # Union alle perioder
        df_final = dfs[0]
        for df in dfs[1:]:
            df_final = df_final.unionByName(df, allowMissingColumns=True)

        # Berikelse
        df_final = enrich_with_kommune_2024(df_final)

        # Cache enriched DataFrame – brukes av kvalitetssjekk og skriving
        df_final.cache()

        print("Datakvalitet:")
        quality = run_data_quality_checks(df_final)

        # Skriv til Standardize – sender last_loaded_timestamp for å avgjøre strategi
        write_to_delta(df_final, target_table, last_loaded_timestamp)
        df_final.unpersist()

        # Merk tabellen med beskrivelse/kategori/frekvens/kilde-metadata
        set_table_metadata(
            target_table, table_id, config_table_name, config_category, config_frequency
        )

        # Kjør tabellspesifikk refine-notebook hvis den finnes
        refine_nb = f"ssb_refine_{table_id}"
        try:
            mssparkutils.notebook.run(refine_nb, 1800, {"TABELLNR": table_id})
            print(f"   {refine_nb} fullført")
        except Exception as e:
            err = str(e)
            if "Fetch notebook content" in err or "NotebookExecutionException" in err:
                print(f"   {refine_nb} finnes ikke – hopper over")
            else:
                print(f"   ADVARSEL: {refine_nb} feilet: {err}")

        result["status"]  = "success"
        result["rows"]    = quality["total_rows"]
        result["periods"] = len(dfs)
        return result

    except Exception as e:
        import traceback
        print(f"\nFEIL: {e}")
        traceback.print_exc()
        result["status"] = "failed"
        result["error"]  = str(e)
    return result

print("process_table OK")

StatementMeta(, eb88c85d-40bd-4a35-85ce-1ceb6f269737, 14, Finished, Available, Finished, False)

process_table OK


In [13]:
# =====================================================================
# HOVEDLOOP – prosesser alle tabeller fra koen
# Går gjennom hver unike tabell i køen og kaller process_table for den.
# Etter at alle er behandlet oppdateres last_loaded_timestamp i ssb_config
# for de som gikk bra, og et sammendrag med resultatet for hver tabell
# skrives ut til slutt.

start_time = time.time()

print(f"Ko: {QUEUE_TABLE}")
print(f"Write mode: {WRITE_MODE}")
print("="*60)

if not spark.catalog.tableExists(QUEUE_TABLE):
    raise RuntimeError(
        f"Ko-tabell '{QUEUE_TABLE}' finnes ikke. "
        "Kjor 03_ssb_oppdateringsdetector og 04_ssb_ingest_landing forst."
    )

# Hent unike tabeller fra koen (deduper pa table_id)
queue_rows = (
    spark.table(QUEUE_TABLE)
    .select("table_id")
    .distinct()
    .collect()
)

# Midlertidig – begrens til én tabell for testing
if DEBUG_TABLE_ID:
    queue_rows = [r for r in queue_rows if r["table_id"] == DEBUG_TABLE_ID]
    print(f"DEBUG: filtrert til tabell {DEBUG_TABLE_ID}")

print(f"Tabeller i kø: {[r['table_id'] for r in queue_rows]}")

if not queue_rows:
    print("Ko er tom – ingen tabeller a prosessere. Avslutter.")
else:
    print(f"Fant {len(queue_rows)} tabeller i koen\n")

    results    = []
    ok_count   = 0
    fail_count = 0

    for row in queue_rows:
        result = process_table(row["table_id"])
        results.append(result)
        if result["status"] == "success":
            ok_count += 1
        else:
            fail_count += 1

    elapsed = int(time.time() - start_time)

    print(f"\n{'='*60}")
    print(f"SAMMENDRAG  ({elapsed}s totalt)")
    print(f"{'='*60}")
    print(f"  Vellykket: {ok_count}")
    print(f"  Feilet:    {fail_count}")
    print()

    for r in results:
        icon = "OK" if r["status"] == "success" else "!!"
        print(
            f"  [{icon}] {r['table_id']}: "
            f"{r['target_table']:<35} "
            f"{r['rows']:>10,} rader, {r['periods']} perioder"
        )

    if fail_count > 0:
        print("\nFeil-detaljer:")
        for r in results:
            if r["status"] == "failed":
                print(f"  {r['table_id']}: {r['error']}")

    # Oppdater last_loaded_timestamp for vellykkede tabeller
    processed_ok = [r["table_id"] for r in results if r["status"] == "success"]
    if processed_ok:
        loaded_ts = datetime.now(timezone.utc).strftime("%Y-%m-%dT%H:%M:%S+00:00")
        update_last_loaded_timestamp(processed_ok, loaded_ts)
    else:
        print("⚠️  Ingen tabeller ble vellykket prosessert – ssb_config ikke oppdatert")

    print("SILVER TRANSFORMASJON FULLFORT")
    print(f"{'='*60}")

StatementMeta(, eb88c85d-40bd-4a35-85ce-1ceb6f269737, 15, Finished, Available, Finished, False)

Ko: statbank_staging.pipeline.ssb_config
Write mode: overwrite
DEBUG: filtrert til tabell 06194
Tabeller i kø: ['06194']
Fant 1 tabeller i koen


Tabell: 06194  ->  ssb.statbank_06194
Snapshot: 2026-06-15
Lookback: deaktivert (full_load=True) – laster alle 21 perioder
Perioder: 21 (2005 -> 2025)
   [10/21] 2014 OK
   [20/21] 2024 OK
   [21/21] 2025 OK
   Bruker 'region_code' som regionskolonne
   Kommune-berikelse: 132,447/144,795 rader mappet (91.5%)
Datakvalitet:
   Rader: 144,795  |  Null: 15,876 (11.0%)  |  Duplikater: 0  |  Ar: 2005-2025
   Skrivemodus: full overwrite (WRITE_MODE=overwrite)
   OPTIMIZE fullført
   Skrevet til ssb.statbank_06194
   ssb_refine_06194 finnes ikke – hopper over

SAMMENDRAG  (110s totalt)
  Vellykket: 1
  Feilet:    0

  [OK] 06194: ssb.statbank_06194                     144,795 rader, 21 perioder
✅ last_loaded_timestamp oppdatert for 1 tabeller: 06194
SILVER TRANSFORMASJON FULLFORT
